# Correction — Exercice 2 ★★☆☆☆ — Votre première vraie table / Exercice 3 ★★★☆☆ — Un champ optionnel avec valeur par défaut

> À ouvrir seulement en cas de blocage ! L'exercice est dans `exercice2_3.ipynb`.

Lancez les cellules dans l'ordre (**Run All**) : les tests doivent tous être ✅.

## Énoncé

### Exercice 2 ★★☆☆☆ — Votre première vraie table

Créez une classe `Category` qui **est** une table en base, avec :
- `id` : entier optionnel, clé primaire, `None` par défaut (générée par la base)
- `name: str`

### Exercice 3 ★★★☆☆ — Un champ optionnel avec valeur par défaut

Modifiez `Category` pour ajouter un champ `description`, de type texte, **optionnel**, qui vaut `None` si rien n'est fourni.

In [ ]:
from sqlmodel import SQLModel, Field, Relationship
from sqlalchemy.orm import clear_mappers

## Correction

In [ ]:
# Ne pas toucher : permet de relancer cette cellule sans erreur.
SQLModel.metadata.clear()
clear_mappers()

# table=True : la classe devient une vraie table "category" en base.
# id vaut None tant que l'objet n'est pas enregistré ; la base le génère.
class Category(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    description: str | None = None  # Exercice 3 : optionnel, None par défaut


### Explication

- `table=True` crée la table `category` en base.
- `id` vaut `None` tant que l'objet n'est pas enregistré : c'est la base qui génère la valeur à l'insertion.
- `str | None = None` rend `description` optionnelle. Pas besoin de `Field()` pour ça.
- L'exercice 3 modifie la même classe : on ajoute le champ dans `Category`, on n'en crée pas une deuxième.

## Vérification

In [ ]:
import pytest

def get_fields(model):
    """Compatible pydantic v1 (SQLModel ancien) et v2 (SQLModel récent)."""
    if hasattr(model, "model_fields"):
        return model.model_fields
    return model.__fields__


def is_table(model) -> bool:
    return getattr(model, "__table__", None) is not None


# ---------------------------------------------------------------------------
# Exercice 2 — Category (première table)
# ---------------------------------------------------------------------------

def test_exercice2_category_existe():
    assert hasattr(ex, "Category"), "La classe Category est introuvable."


def test_exercice2_category_est_une_table():
    assert is_table(ex.Category), "Category doit être déclarée avec table=True."


def test_exercice2_category_a_les_bons_champs():
    fields = get_fields(ex.Category)
    for champ in ("id", "name"):
        assert champ in fields, f"Le champ '{champ}' est manquant sur Category."


def test_exercice2_category_instanciation():
    c = ex.Category(name="Romans")
    assert c.name == "Romans"
    assert c.id is None  # pas encore enregistrée en base


# ---------------------------------------------------------------------------
# Exercice 3 — Category.description (champ optionnel)
# ---------------------------------------------------------------------------

def test_exercice3_category_a_description():
    fields = get_fields(ex.Category)
    assert "description" in fields, (
        "Le champ 'description' est manquant sur Category (exercice 3)."
    )


def test_exercice3_description_est_optionnelle():
    c = ex.Category(name="Romans")  # sans description
    assert c.description is None, (
        "description doit valoir None par défaut si elle n'est pas fournie."
    )


def test_exercice3_description_peut_etre_fournie():
    c = ex.Category(name="Romans", description="Fiction narrative")
    assert c.description == "Fiction narrative"

# --- Lancement des tests ----------------------------------------------------
import types

ex = types.SimpleNamespace(**globals())
_ok = 0
_tests = [(nom, f) for nom, f in list(globals().items()) if nom.startswith("test_")]
for _nom, _test in _tests:
    try:
        _test()
        print("✅", _nom)
        _ok += 1
    except Exception as erreur:
        print("❌", _nom, "→", str(erreur) or type(erreur).__name__)
print(f"\n{_ok}/{len(_tests)} tests réussis" + (" — bravo !" if _ok == len(_tests) else ""))
